# Análise Econômica e Machine Learning
## Aula 01 — Laboratório: a primeira coleta de dados econômicos

**Universidade Católica de Pernambuco** · Curso de Economia, 8º período
Prof. Dr. Flavius Sodré

---

### Como funciona a aula de hoje

Hoje **você programa a aula inteira**.
O notebook é o roteiro.

| | |
|---|---|
| **Duração** | 4 horas, em 6 blocos cronometrados |
| **Formato** | Duplas. Um digita, o outro lê a documentação — e vocês trocam a cada bloco |
| **Entrega** | Este notebook preenchido, ao final da aula |

### As três regras

1. **Tente antes de perguntar.** Leia a mensagem de erro inteira primeiro — ela quase sempre diz o que fazer.
2. **Cinco minutos travado é o limite.** 
3. **Rode as células em ordem**, de cima para baixo. Pular célula é a causa nº 1 de erro nesta aula.

### O que você vai saber fazer ao final

Baixar séries econômicas direto de APIs públicas (Banco Central, IBGE, Banco Mundial),
juntar séries de frequências diferentes num mesmo quadro, e — o mais importante —
**dizer o que há de errado com os dados que você acabou de baixar**.

---
# Bloco 0 — Ambiente no ar
### ⏱️ 15 minutos

**Meta do bloco:** todo mundo com o notebook rodando e a rede testada.

Rode as três células abaixo, na ordem.

In [ ]:
# Bloco 0 — rode esta célula e confira a saída.
import sys, platform
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 12)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Python  :", sys.version.split()[0], "|", platform.system())
print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)
print("requests:", requests.__version__)
print("\nAmbiente pronto.")

In [ ]:
# Bloco 0 — teste de rede.
MODO_OFFLINE = False

def testar_rede():
    global MODO_OFFLINE
    try:
        r = requests.get(
            "https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados/ultimos/1",
            params={"formato": "json"}, timeout=20)
        r.raise_for_status()
        print("[OK] A API do Banco Central respondeu:", r.json())
        print("Voce vai trabalhar com dados reais.")
    except Exception as e:
        MODO_OFFLINE = True
        print("[SEM REDE]", type(e).__name__, "-", e)
        print("O laboratorio segue com o PLANO B (dados sinteticos).")
        print("Avise o professor, mas NAO pare: os exercicios funcionam igual.")

testar_rede()

In [ ]:
# Bloco 0 — mecanismo de contingência. Leia, mas não precisa alterar.
#
# Toda coleta deste notebook passa por `com_plano_b(...)`. Se a API responder,
# você usa o dado real. Se não responder, o notebook gera uma série SINTÉTICA
# só para o exercício continuar rodando.
#
# ATENÇÃO: dado sintético é dado inventado. Ele serve para você praticar a
# mecânica (limpar, juntar, medir buraco). Ele NÃO serve para interpretar
# economia. O aviso aparece impresso sempre que o Plano B for acionado.

def serie_sintetica(inicio, periodos, freq, media, desvio, semente):
    rng = np.random.default_rng(semente)
    idx = pd.date_range(inicio, periods=periodos, freq=freq)
    valores = media + rng.normal(0, desvio, periodos)
    return pd.Series(valores.round(2), index=idx, name="valor")

def sidra_sintetico(n=12, semente=11, rotulo_periodo="Mes", media=0.45, desvio=0.3):
    """Imita a estrutura de resposta do SIDRA, para o exercicio nao travar."""
    rng = np.random.default_rng(semente)
    periodos = pd.period_range("2024-01", periods=n, freq="M").strftime("%Y%m")
    df = pd.DataFrame({
        "NC": ["1"] * n,
        "NN": ["Brasil"] * n,
        "V": np.round(media + rng.normal(0, desvio, n), 2).astype(str),
        "D3C": list(periodos),
        "D3N": list(periodos),
    })
    rotulos = {"NC": "Nivel Territorial (Codigo)", "NN": "Nivel Territorial",
               "V": "Valor", "D1C": "Brasil (Codigo)", "D1N": "Brasil",
               "D2C": "Variavel (Codigo)", "D2N": "Variavel",
               "D3C": f"{rotulo_periodo} (Codigo)", "D3N": rotulo_periodo}
    return df, rotulos

def wb_sintetico(semente, media=2.0, desvio=3.0, inicio=2000, fim=2024, anos_vazios=2):
    """Imita o World Bank, inclusive a lacuna dos anos mais recentes."""
    rng = np.random.default_rng(semente)
    anos = list(range(inicio, fim + 1))
    v = np.round(rng.normal(media, desvio, len(anos)), 2)
    v[-anos_vazios:] = np.nan          # o ultimo dado costuma nao ter chegado ainda
    return pd.Series(v, index=anos)

def com_plano_b(rotulo, coleta, contingencia):
    """Tenta a API; se falhar, devolve a contingencia com aviso bem visivel.
    Funciona tanto para uma Series quanto para o par (DataFrame, rotulos)."""
    global MODO_OFFLINE
    def _tamanho(x):
        return len(x[0]) if isinstance(x, tuple) else len(x)
    if not MODO_OFFLINE:
        try:
            resultado = coleta()
            print(f"[REAL] {rotulo}: {_tamanho(resultado)} observacoes vindas da API.")
            return resultado
        except Exception as e:
            print(f"[FALHOU] {rotulo}: {type(e).__name__} - {e}")
            MODO_OFFLINE = True
    resultado = contingencia()
    print("=" * 72)
    print(f"[PLANO B] {rotulo}: DADOS SINTETICOS (inventados).")
    print("Pratique a mecanica, mas nao tire conclusoes economicas daqui.")
    print("=" * 72)
    return resultado

print("Plano B carregado.")

> ### ✅ Checkpoint 0
> Você só avança quando as três células acima rodaram sem erro vermelho.
> Se apareceu `[SEM REDE]`, tudo bem — siga normalmente.

---
# Bloco 1 — Banco Central: sua primeira série
### ⏱️ 40 minutos

O Banco Central publica milhares de séries pelo **SGS**. Cada série tem um número.
Você vai buscar três delas.

| Código | Série | Frequência |
|---|---|---|
| 433 | IPCA — variação % mensal | mensal |
| 432 | Meta Selic definida pelo Copom (% a.a.) | diária |
| 1 | Dólar comercial (venda) | diária |
| 189 | IGP-M — variação % mensal | mensal |

Para achar outros códigos, o catálogo está em
<https://www3.bcb.gov.br/sgspub/> (Consulta de séries → busque pelo nome).

In [ ]:
# Bloco 1 — função pronta. Leia com atenção: você vai reutilizá-la o dia todo.

def sgs(codigo, inicio="01/01/2015", fim=None):
    """Baixa uma serie do SGS (Sistema Gerenciador de Series Temporais) do Banco Central.

    codigo : int    - o numero da serie no SGS (ex.: 433 = IPCA mensal)
    inicio : str    - data inicial no formato dd/mm/aaaa
    retorna: pd.Series indexada por data
    """
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
    params = {"formato": "json", "dataInicial": inicio}
    if fim:
        params["dataFinal"] = fim
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
    return df.set_index("data")["valor"].sort_index()

print("Funcao sgs() pronta.")

In [ ]:
# Bloco 1 — EXEMPLO RESOLVIDO. Rode e observe o formato do resultado.
ipca = com_plano_b(
    "IPCA mensal (SGS 433)",
    lambda: sgs(433, inicio="01/01/2015"),
    lambda: serie_sintetica("2015-01-01", 120, "MS", media=0.45, desvio=0.30, semente=1),
)
display(ipca.head())
print("\nPeriodo:", ipca.index.min().date(), "ate", ipca.index.max().date())
print("Observacoes:", len(ipca))

In [ ]:
# Bloco 1 — TAREFA 1.1
# Baixe a Selic (codigo 432) a partir de 01/01/2015, seguindo o exemplo acima.
# Dica: a Selic e DIARIA. Nao estranhe se vierem milhares de linhas.

selic = com_plano_b(
    "Selic meta (SGS 432)",
    lambda: ...,          # <-- COMPLETE: chame sgs() com o codigo certo
    lambda: serie_sintetica("2015-01-01", 2600, "B", media=9.0, desvio=0.05, semente=2),
)
display(selic.head())
print("Observacoes:", len(selic))

In [ ]:
# Bloco 1 — TAREFA 1.2
# Agora o dolar (codigo 1), tambem a partir de 01/01/2015.

dolar = com_plano_b(
    "Dolar comercial (SGS 1)",
    lambda: ...,          # <-- COMPLETE
    lambda: serie_sintetica("2016-01-04", 2340, "B", media=4.5, desvio=0.6, semente=3),
)
display(dolar.tail())

In [ ]:
# Bloco 1 — TAREFA 1.3
# Faca um grafico de linha do IPCA mensal.
# Dica: uma Series do pandas ja sabe se desenhar -> serie.plot()

# <-- COMPLETE: desenhe o ipca
ax = ...
plt.title("IPCA — variacao % mensal")
plt.ylabel("% no mes")
plt.xlabel("")
plt.show()

In [ ]:
# Bloco 1 — TAREFA 1.4
# Responda NO CODIGO, imprimindo os valores:
#   a) quantas observacoes tem cada serie?
#   b) qual o primeiro e o ultimo dia de cada uma?

for nome, s in [("IPCA", ipca), ("Selic", selic), ("Dolar", dolar)]:
    pass  # <-- COMPLETE: imprima nome, len(s), s.index.min(), s.index.max()

> ### ✅ Checkpoint 1 — responda aqui (dê dois cliques para editar)
>
> **1.** As três séries têm o mesmo número de observações? Por quê?
>
> *sua resposta:*
>
> **2.** Na série 433, o valor `0.42` significa que o IPCA *é* 0,42 — ou que ele *subiu* 0,42?
> Qual a diferença prática entre nível e variação?
>
> *sua resposta:*
>
> **3.** Se você precisasse do IPCA acumulado em 12 meses, buscaria outra série ou calcularia
> a partir desta? (Não precisa fazer — só decidir e justificar.)
>
> *sua resposta:*

---
# Bloco 2 — IBGE: a mesma inflação por outra porta
### ⏱️ 40 minutos

O IPCA é **produzido pelo IBGE** — o Banco Central apenas republica. Neste bloco você
busca o mesmo indicador na fonte primária, pela API **SIDRA**, e compara.

A consulta do SIDRA é montada como um caminho:

```
/t/1737 /n1/all /v/63 /p/last12
  │       │      │     └── período: os 12 mais recentes
  │       │      └──────── variável: 63 = variação mensal
  │       └─────────────── nível territorial: n1 = Brasil
  └─────────────────────── tabela 1737 = IPCA série histórica
```

In [ ]:
# Bloco 2 — função pronta para a API SIDRA (IBGE).

def sidra_bruto(caminho):
    """Consulta a API SIDRA e devolve (DataFrame, rotulos).

    caminho: o trecho da consulta, ex.: "/t/1737/n1/all/v/63/p/last12"
             t = tabela, n1 = Brasil, v = variavel, p = periodo

    A PRIMEIRA linha que o SIDRA devolve nao e dado: e a legenda das colunas.
    Por isso ela sai separada, em `rotulos`.
    """
    url = "https://apisidra.ibge.gov.br/values" + caminho
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    dados = r.json()
    if not dados:
        raise ValueError("O SIDRA devolveu uma resposta vazia.")
    rotulos, linhas = dados[0], dados[1:]
    if not linhas:
        raise ValueError("O SIDRA devolveu apenas o cabecalho (consulta sem dados).")
    return pd.DataFrame(linhas), rotulos

print("Funcao sidra_bruto() pronta.")

In [ ]:
# Bloco 2 — EXEMPLO RESOLVIDO. Rode e olhe a estrutura.
df_ipca, rotulos = com_plano_b(
    "IPCA via SIDRA (tabela 1737)",
    lambda: sidra_bruto("/t/1737/n1/all/v/63/p/last12"),
    lambda: sidra_sintetico(n=12, semente=11),
)
print("Colunas devolvidas e o que significam:\n")
for k, v in rotulos.items():
    print(f"  {k:6s} -> {v}")
print()
display(df_ipca.head(3))

In [ ]:
# Bloco 2 — TAREFA 2.1
# Olhando os rotulos impressos acima, descubra:
#   - qual coluna guarda o CODIGO do periodo (algo como "202401")
#   - qual coluna guarda o VALOR
# e monte uma Series indexada por data.

col_periodo = "..."   # <-- COMPLETE (dica: termina em "C" e o rotulo fala em "Mes")
col_valor = "..."     # <-- COMPLETE

serie = pd.Series(
    pd.to_numeric(df_ipca[col_valor], errors="coerce").values,
    index=pd.to_datetime(df_ipca[col_periodo], format="%Y%m"),
    name="ipca_ibge",
)
display(serie)

In [ ]:
# Bloco 2 — TAREFA 2.2
# Repita a consulta, mas pedindo os ULTIMOS 24 meses em vez de 12.
# Depois transforme em Series do mesmo jeito.

df24, rot24 = com_plano_b(
    "IPCA via SIDRA, 24 meses",
    lambda: sidra_bruto("..."),   # <-- COMPLETE o caminho
    lambda: sidra_sintetico(n=24, semente=12),
)

ipca_ibge = pd.Series(
    pd.to_numeric(df24["V"], errors="coerce").values,
    index=pd.to_datetime(df24["D3C"], format="%Y%m"),
    name="ipca_ibge",
).sort_index()
print(len(ipca_ibge), "meses")
display(ipca_ibge.head())

In [ ]:
# Bloco 2 — TAREFA 2.3
# Agora um indicador diferente: a taxa de desocupacao da PNAD Continua.
#   tabela 6381, nivel n1 (Brasil), variavel 4099, ultimos 12 trimestres moveis
# Monte o caminho e rode. Se a consulta falhar, LEIA o erro e inspecione os rotulos.

df_desoc, rot_desoc = com_plano_b(
    "Desocupacao PNAD (tabela 6381)",
    lambda: sidra_bruto("..."),   # <-- COMPLETE
    lambda: sidra_sintetico(n=12, semente=13, rotulo_periodo="Trimestre Movel",
                            media=8.0, desvio=1.2),
)

for k, v in rot_desoc.items():
    print(f"  {k:6s} -> {v}")
display(df_desoc.head(3))

In [ ]:
# Bloco 2 — TAREFA 2.4
# Transforme a desocupacao numa Series. ATENCAO: o periodo aqui e TRIMESTRE MOVEL,
# nao mes. Olhe o formato do codigo de periodo antes de converter.

desocupacao = ...   # <-- COMPLETE
display(desocupacao)

> ### ✅ Checkpoint 2 — a prova dos nove
>
> Rode a célula abaixo. Ela compara o IPCA que você pegou **no Banco Central** com o
> que você pegou **no IBGE**, para os mesmos meses.

In [ ]:
# Bloco 2 — Checkpoint 2
# As duas series falam de MES, mas gravam a data de formas diferentes.
# Convertendo as duas para periodo mensal, elas passam a alinhar.
a = ipca.copy();      a.index = a.index.to_period("M")
b = ipca_ibge.copy(); b.index = b.index.to_period("M")

comparacao = pd.DataFrame({"bcb_433": a, "ibge_1737": b}).dropna()
comparacao["diferenca"] = (comparacao["bcb_433"] - comparacao["ibge_1737"]).round(4)
display(comparacao.tail(12))
print("\nMeses comparaveis:", len(comparacao))
print("Maior diferenca absoluta:", comparacao["diferenca"].abs().max())

> **Pergunta:** a diferença deu zero (ou quase)? O que isso te diz sobre "fonte" de dado?
>
> *sua resposta:*
>
> E se tivesse dado diferente de zero, quais seriam as três explicações mais prováveis?
>
> *sua resposta:*

---
# ☕ Intervalo — 15 minutos

---
# Bloco 3 — Juntar o que não se encaixa
### ⏱️ 40 minutos

Você tem três séries **de frequências diferentes**: mensal, diária e diária.
Ninguém analisa nada assim. Este bloco é a parte do trabalho que ninguém mostra
em aula teórica — e que consome a maior parte do tempo de um analista de verdade.

In [ ]:
# Bloco 3 — EXPERIMENTO 3.0 (rode e olhe o estrago)
# Juncao INGENUA: reduzir a serie diaria para mensal e juntar com o IPCA.
selic_ingenua = selic.resample("ME").mean()

print("Ultimas datas do IPCA :", list(ipca.index[-3:]))
print("Ultimas datas da Selic:", list(selic_ingenua.index[-3:]))
display(pd.DataFrame({"ipca": ipca, "selic": selic_ingenua}).tail(6))
print("\nPor que quase tudo virou NaN?")

> **Pare e pense (2 min).** As duas séries são mensais. Mesmo assim a junção falhou.
> Olhe as datas impressas acima: o IPCA está marcado no dia **1º** do mês, e a Selic
> reamostrada no **último** dia. Para o pandas, `2024-01-01` e `2024-01-31` são dois
> instantes diferentes — ele não sabe que ambos querem dizer "janeiro de 2024".
>
> A correção é dizer explicitamente que a unidade é o **mês**, com `.to_period("M")`.

In [ ]:
# Bloco 3 — TAREFA 3.1
# Reduza as series diarias para MENSAIS.
#   - Selic: use a MEDIA do mes (e uma taxa, faz sentido tirar media)
#   - Dolar: use o ULTIMO valor do mes (cotacao de fechamento)
# Dica: serie.resample("ME").mean()  /  serie.resample("ME").last()

selic_m = ...   # <-- COMPLETE
dolar_m = ...   # <-- COMPLETE

# depois de reamostrar, padronize as tres series em periodo mensal:
ipca_m = ipca.copy();   ipca_m.index = ipca_m.index.to_period("M")
selic_m.index = selic_m.index.to_period("M")
dolar_m.index = dolar_m.index.to_period("M")

print("Selic mensal:", len(selic_m), "| Dolar mensal:", len(dolar_m))
display(pd.DataFrame({"selic": selic_m, "dolar": dolar_m}).tail())

In [ ]:
# Bloco 3 — TAREFA 3.2
# Junte as tres series mensais num unico DataFrame chamado `painel`.
# Dica: pd.DataFrame({...}) alinha automaticamente pelo indice.

painel = ...   # <-- COMPLETE
display(painel.tail(8))
print("\nFormato:", painel.shape)

In [ ]:
# Bloco 3 — TAREFA 3.3
# Onde estao os buracos? Conte os valores ausentes por coluna
# e mostre as linhas que tem pelo menos um ausente.

print(...)          # <-- COMPLETE: contagem de NaN por coluna
display(...)        # <-- COMPLETE: as linhas incompletas

> ### ✅ Checkpoint 3
>
> **1.** Quantas linhas do painel estão incompletas? Onde elas se concentram — no começo,
> no fim, ou espalhadas?
>
> *sua resposta:*
>
> **2.** Um `NaN` no fim da série e um `NaN` no meio da série têm a mesma causa?
> O que cada um provavelmente significa?
>
> *sua resposta:*

---
# Bloco 4 — O dado mente? Cobertura, defasagem e viés
### ⏱️ 40 minutos

Este é o bloco conceitual da aula — mas você o faz **medindo**, não ouvindo.
A pergunta que organiza tudo: *o que este dado não me conta?*

In [ ]:
# Bloco 4 — EXPERIMENTO 4.1 (rode e observe)
# O que acontece com a media quando voce "conserta" os buracos no automatico?

antes = painel["dolar"].mean()
zero = painel["dolar"].fillna(0).mean()
frente = painel["dolar"].ffill().mean()
sem = painel["dolar"].dropna().mean()

print(f"media ignorando NaN (padrao do pandas) : {antes:.4f}")
print(f"media apos fillna(0)                    : {zero:.4f}")
print(f"media apos ffill()                      : {frente:.4f}")
print(f"media apos dropna()                     : {sem:.4f}")
print("\nQual delas esta ERRADA? Por que?")

> **Responda antes de seguir:** três dessas médias são iguais e uma é diferente.
> Qual está errada, e por quê? E por que `ffill()` não mudou nada?
>
> *sua resposta:*

In [ ]:
# Bloco 4 — TAREFA 4.2
# Mede a DEFASAGEM de cada serie: quantos dias entre a ultima observacao
# disponivel e hoje?

hoje = pd.Timestamp.today().normalize()
for nome, s in [("IPCA (mensal)", ipca), ("Selic (diaria)", selic), ("Dolar (diario)", dolar)]:
    pass  # <-- COMPLETE: imprima nome, ultima data e (hoje - ultima data).days

In [ ]:
# Bloco 4 — função pronta para a API do Banco Mundial.

def world_bank(pais, indicador, inicio=2000, fim=2024):
    """Baixa um indicador do World Bank.

    pais      : codigo ISO3, ex. "BRA", "ARG", "CHL", "MEX"
    indicador : ex. "NY.GDP.MKTP.KD.ZG" (crescimento do PIB, % ao ano)
    """
    url = f"https://api.worldbank.org/v2/country/{pais}/indicator/{indicador}"
    params = {"format": "json", "per_page": 500, "date": f"{inicio}:{fim}"}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    corpo = r.json()
    if len(corpo) < 2 or corpo[1] is None:
        raise ValueError("O Banco Mundial nao devolveu dados para essa combinacao.")
    df = pd.DataFrame(corpo[1])[["date", "value"]]
    df["date"] = df["date"].astype(int)
    return df.sort_values("date").set_index("date")["value"]

print("Funcao world_bank() pronta.")

In [ ]:
# Bloco 4 — TAREFA 4.3
# Compare o crescimento do PIB do Brasil com o de outro pais da America Latina.
# Indicador: "NY.GDP.MKTP.KD.ZG" (crescimento anual do PIB, %)

pib_br = com_plano_b(
    "PIB Brasil (World Bank)",
    lambda: world_bank("BRA", "NY.GDP.MKTP.KD.ZG", 2000, 2024),
    lambda: wb_sintetico(semente=7, media=2.0, desvio=3.0),
)

pib_outro = com_plano_b(
    "PIB (outro pais)",
    lambda: ...,   # <-- COMPLETE: escolha ARG, CHL, MEX, COL ou PER
    lambda: wb_sintetico(semente=8, media=1.5, desvio=4.0),
)

comp = pd.DataFrame({"Brasil": pib_br, "Outro": pib_outro})
display(comp.tail(10))
print("\nAnos sem dado:")
print(comp.isna().sum())

In [ ]:
# Bloco 4 — TAREFA 4.4
# A desocupacao da PNAD e TRIMESTRAL; o IPCA e MENSAL.
# Sem escrever codigo de juncao ainda, responda no print: quantas observacoes
# de desocupacao existem por ano, e quantas de IPCA?

print("Obs. de desocupacao:", len(desocupacao))
print("Obs. de IPCA        :", len(ipca_ibge))
print("\nSe voce juntasse as duas mes a mes, quantas linhas ficariam vazias?")

> ### ✅ Checkpoint 4 — a ficha de vieses
>
> Escreva **três frases**, uma para cada item. Frases curtas, com o número que você mediu.
>
> **1. Cobertura** — o que estes dados *não* cobrem?
>
> *sua resposta:*
>
> **2. Defasagem** — quanto tempo depois do fato o dado aparece?
>
> *sua resposta:*
>
> **3. Comparabilidade** — o que impede comparar direto Brasil com o outro país,
> ou o IPCA com a desocupação?
>
> *sua resposta:*

---
# Bloco 5 — Desafio em dupla: seu primeiro painel
### ⏱️ 45 minutos

Agora sem passo a passo. Vocês montam, do zero, um pequeno painel de dados
econômicos — e o entregam **documentado**.

### O que entregar

1. **Três indicadores** de sua escolha, de pelo menos **duas fontes diferentes**
   (BCB, IBGE ou Banco Mundial).
2. **Um `DataFrame` único**, com as séries alinhadas na mesma frequência.
3. **Um gráfico** que faça sentido — com título, eixo nomeado e unidade.
4. **A ficha técnica** preenchida (a última célula deste notebook).

### Como isso é avaliado

| Critério | O que estamos olhando |
|---|---|
| Coleta | O código roda do começo ao fim sem intervenção |
| Tratamento | As frequências foram alinhadas com uma decisão justificada |
| Honestidade | A ficha técnica declara os limites reais dos dados |
| Leitura | O gráfico comunica algo, e a legenda diz o quê |

> Não vale escolher três séries do mesmo lugar e com a mesma frequência: o desafio
> é justamente conciliar fontes que não conversam.

In [ ]:
# Bloco 5 — ESPACO DE TRABALHO
# Escreva aqui a coleta dos seus tres indicadores.
# Reaproveite sgs(), sidra_bruto() e world_bank() — nao comece do zero.






In [ ]:
# Bloco 5 — seu painel unico (DataFrame)





In [ ]:
# Bloco 5 — seu grafico





---
## 📋 Ficha técnica dos dados — preencha antes de entregar

Copie a tabela e complete uma linha por indicador.

| | Indicador 1 | Indicador 2 | Indicador 3 |
|---|---|---|---|
| **Nome do indicador** | | | |
| **Quem produz** (não quem republica) | | | |
| **Endereço da API** | | | |
| **Frequência original** | | | |
| **Frequência depois do seu tratamento** | | | |
| **Como você alinhou** (média, último, soma…) | | | |
| **Período coberto** | | | |
| **Defasagem de divulgação** | | | |
| **O que este dado NÃO cobre** | | | |

### Uma última pergunta, para responder em duas linhas

Se um jornalista pegasse o seu gráfico e escrevesse uma manchete a partir dele,
qual seria a manchete **errada** que ele poderia escrever — e que o seu dado
não sustenta?

*sua resposta:*

---